# RAG

In [12]:
import requests
import xml.etree.ElementTree as ET
import json
import os
import time

API = "http://export.arxiv.org/api/query"
THEME_KEYWORDS = ["exoplanet", "planet detection", "habitability", "TRAPPIST", "TESS"]
# THEME_KEYWORDS = ["exoplanet", "TRAPPIST"]
OUTPUT_FILE = "exoplanet_data.json"
MAX_DATA_SIZE = 2000

def get_exoplanet_data_xml(start=0, batch_size=100):
    search_terms = " OR ".join([f'all:{keyword}' for keyword in THEME_KEYWORDS])
    query = f"search_query=({search_terms})&start={start}&max_results={batch_size}"
    url = f"{API}?{query}"
    
    response = requests.get(url)
    response.raise_for_status()
    
    root = ET.fromstring(response.content)
    return root

def read_xml(xml_root: ET.Element, output):
    namespace = {"atom": "http://www.w3.org/2005/Atom"}
    rows_added = 0
    
    for entry in xml_root.findall('atom:entry', namespace):
        authors = entry.findall('atom:author', namespace)
        
        categories = entry.findall('atom:category', namespace)
        categories_list = [cat.get('term') for cat in categories] if categories else []
        
        title_elem = entry.find('atom:title', namespace)
        id_elem = entry.find('atom:id', namespace)
        published_elem = entry.find('atom:published', namespace)
        summary_elem = entry.find('atom:summary', namespace)
        
        if title_elem is None or id_elem is None or published_elem is None or summary_elem is None:
            continue
        
        author_names = []
        for author in authors:
            name_elem = author.find('atom:name', namespace)
            if name_elem is not None and name_elem.text:
                author_names.append(name_elem.text)
        
        row = {
            "title": title_elem.text.strip() if title_elem.text else "",
            "id": id_elem.text if id_elem.text else "",
            "published": published_elem.text if published_elem.text else "",
            "summary": summary_elem.text.strip() if summary_elem.text else "",
            "authors": author_names,
            "categories": categories_list
        }
        output.append(row)
        rows_added += 1
        
    return rows_added

def fetch_exoplanet_data(output_file=OUTPUT_FILE):
    batch_size = 100
    total_rows = 0
    data = []
    
    print("Start fetching exoplanet data...")
    
    while total_rows < MAX_DATA_SIZE:
        print(f"Fetching data from {total_rows} to {total_rows + batch_size}")
        
        try:
            xml_root = get_exoplanet_data_xml(total_rows, batch_size)
            rows_added = read_xml(xml_root, data)
            
            if rows_added == 0:
                print("No more data found, exiting...")
                break
                
            total_rows += rows_added
            print(f"Added {rows_added} rows to {output_file}, total rows: {total_rows}")
            time.sleep(1)
            
        except Exception as e:
            print(f"Error fetching data: {e}")
            break
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    
    print(f"Data saved to {output_file}")
    
    return data

articles = fetch_exoplanet_data()

Start fetching exoplanet data...
Fetching data from 0 to 100
Added 100 rows to exoplanet_data.json, total rows: 100
Fetching data from 100 to 200
Added 100 rows to exoplanet_data.json, total rows: 200
Fetching data from 200 to 300
Added 100 rows to exoplanet_data.json, total rows: 300
Fetching data from 300 to 400
Added 100 rows to exoplanet_data.json, total rows: 400
Fetching data from 400 to 500
Added 100 rows to exoplanet_data.json, total rows: 500
Fetching data from 500 to 600
Added 100 rows to exoplanet_data.json, total rows: 600
Fetching data from 600 to 700
Added 100 rows to exoplanet_data.json, total rows: 700
Fetching data from 700 to 800
No more data found, exiting...
Data saved to exoplanet_data.json


In [13]:
print(f"Всего загружено публикаций: {len(articles)}")
print(f"Первые 3 публикации:")

for i, article in enumerate(articles[:3]):
    print(f"\n{i+1}. {article['title']}")
    print(f"   Авторы: {', '.join(article['authors'][:3])}{'...' if len(article['authors']) > 3 else ''}")
    print(f"   Категории: {', '.join(article['categories'])}")
    print(f"   Опубликовано: {article['published'][:10]}")
    print(f"   Аннотация: {article['summary'][:200]}...")


Всего загружено публикаций: 700
Первые 3 публикации:

1. TESS Habitable Zone Star Catalog
   Авторы: L. Kaltenegger, J. Pepper, K. Stassun...
   Категории: astro-ph.EP
   Опубликовано: 2019-03-27
   Аннотация: We present the Transiting Exoplanet Survey Satellite (TESS) Habitable Zone
Stars Catalog, a list of 1822 nearby stars with a TESS magnitude brighter than
T = 12 and reliable distances from Gaia DR2, a...

2. Around which stars can TESS detect Earth-like planets? The Revised TESS
  Habitable Zone Catalog
   Авторы: L. Kaltenegger, J. Pepper, P. M. Christodoulou...
   Категории: astro-ph.EP, astro-ph.IM, astro-ph.SR
   Опубликовано: 2021-01-19
   Аннотация: In the search for life in the cosmos, NASA's Transiting Exoplanet Survey
Satellite (TESS) mission has already monitored about 74% of the sky for
transiting extrasolar planets, including potentially ha...

3. Analyzing the Habitable Zones of Circumbinary Planets Using Machine
  Learning
   Авторы: Zhihui Kong, Jonathan H. Jiang, 

In [ ]:
from collections import Counter

# Подсчитаем частоту категорий
all_categories = []
for article in articles:
    all_categories.extend(article['categories'])

# - atstro-ph.EP - Earth and planetary physics
# - astro-ph.SR - Sollar ans Stellar astrophysics
# - astro-ph.IM - Instrumentation and Methods for Astrophysics
# - astro-ph.GA - Astrophysics of Galaxies

category_counts = Counter(all_categories)
print("Топ-10 категорий публикаций:")
for category, count in category_counts.most_common(10):
    print(f"  {category}: {count} публикаций")


Топ-10 категорий публикаций:
  astro-ph.EP: 668 публикаций
  astro-ph.SR: 173 публикаций
  astro-ph.IM: 159 публикаций
  astro-ph.GA: 20 публикаций
  astro-ph: 16 публикаций
  physics.ao-ph: 14 публикаций
  cs.LG: 11 публикаций
  physics.space-ph: 5 публикаций
  physics.geo-ph: 4 публикаций
  physics.pop-ph: 3 публикаций
